In [22]:
import torch
torch.cuda.get_device_name(0)

'Tesla T4'

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import sys
sys.path.append('/content/drive/MyDrive/optimized_kan')

In [25]:
from kan import *
import torch
from torch.quantization import QuantStub, DeQuantStub
import matplotlib.pyplot as plt
import numpy as np
import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# create a KAN: 2D inputs, 1D output, and 5 hidden neurons. cubic spline (k=3), 5 grid intervals (grid=5).

cuda


Create dataset

In [26]:
from kan.utils import create_dataset
from torch.distributions.normal import Normal

eps = 1e-6

# create dataset f(x,y) = exp(sin(pi*x)+y^2)
f = lambda x: (
    x[:, [0]].clamp(min=eps) * Normal(0, 1).cdf(
        (torch.log((x[:, [0]] / x[:, [1]].clamp(min=eps)).clamp(min=eps)) +
         (x[:, [3]] + 0.5 * x[:, [4]]**2) * x[:, [2]]) /
        (x[:, [4]].clamp(min=eps) * torch.sqrt(x[:, [2]].clamp(min=eps)) + eps)
    ) -
    x[:, [1]].clamp(min=eps) * torch.exp(-x[:, [3]] * x[:, [2]]) *
    Normal(0, 1).cdf(
        (torch.log((x[:, [0]] / x[:, [1]].clamp(min=eps)).clamp(min=eps)) +
         (x[:, [3]] - 0.5 * x[:, [4]]**2) * x[:, [2]]) /
        (x[:, [4]].clamp(min=eps) * torch.sqrt(x[:, [2]].clamp(min=eps)) + eps)
    )
)

In [27]:
# WandB Specific
entity = "hpml_project_spring25"
project = "Profiling_Speedups_10000_inference"

# Dataset Specific
n_var = 5
train_num = 10000
test_num = 10000

#Model Specific
seed = 42
grid_size = 16
grid_range = [-1, 1]
spline_order = 3
hidden_dim = [11, 11]
out_dim = 1

# Training Specfic
batch_size = -1
training_steps = 1000
opt = "Adam"
lr = 0.001



dataset = create_dataset(f, n_var=n_var, train_num=train_num, test_num=test_num, device=device)
dataset['train_input'].shape, dataset['train_label'].shape


(torch.Size([10000, 5]), torch.Size([10000, 1]))

Train KAN with sparsity regularization

In [28]:
# train the kan_model
# Initialize W&B project
run = wandb.init(
    entity=entity,
    project=project,
    name="Vanilla KAN",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = KAN(width=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid=wandb.config.grid_size,
                grid_range=wandb.config.grid_range,
                k=wandb.config.spline_order,
                seed=wandb.config.seed,
                device=device)
model(dataset['train_input'][[0]])

kan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

/content/drive/MyDrive/optimized_kan/kan/MultKAN.py:819: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  self.subnode_actscale.append(torch.std(x, dim=0).detach())
/content/drive/MyDrive/optimized_kan/kan/MultKAN.py:829: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  input_range = torch.std(preacts, dim=0) + 0.1
/content/drive/MyDrive/optimized_kan/kan/MultKAN.py:830: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  output_range_spline = torch.std(postacts_numer

checkpoint directory created: ./model
saving model version 0.0


| train_loss: 4.80e-02 | test_loss: 7.70e-02 | reg: 7.92e+00 |: 100%|█| 1000/1000 [00:43<00:00, 23.1


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     124.519ms       155.25%     124.519ms      41.506ms           0 b           0 b           0 b           0 

cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▆▁▂▁▁▂▃▁▁▂█▂▃▂▁▂▅▁▃▁▁▂▁▇▅█▄▁▁▁▁▂▂▂▂▂▂▁▅▇
inference_time,▁
reg,██▆▆▆▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
train_loss,█▇▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,██▇▆▆▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),555.26476


In [29]:
run = wandb.init(
    entity=entity,
    project=project,
    name="Vanilla KAN Compiled",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})


model = KAN(width=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid=wandb.config.grid_size,
                grid_range=wandb.config.grid_range,
                k=wandb.config.spline_order,
                seed=wandb.config.seed,
                device=device).speed(compile=True)

model(dataset['train_input'][[0]])

compiled_kan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})
run.finish()


checkpoint directory created: ./model
saving model version 0.0


W0512 08:37:32.774000 171 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


setting lamb=0. If you want to set lamb > 0, set self.save_act=True


| train_loss: 1.43e-02 | test_loss: 7.19e-02 | reg: 0.00e+00 |: 100%|█| 1000/1000 [00:34<00:00, 28.9


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     141.610ms       200.11%     141.610ms      47.203ms           0 b           0 b           0 b           0 

cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁
inference_time,▁
reg,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇███
train_loss,█▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,██▇▅▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),601.07151


In [30]:
run = wandb.init(
    entity=entity,
    project=project,
    name="Vanilla KAN Compiled w/ Options",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = KAN(width=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid=wandb.config.grid_size,
                grid_range=wandb.config.grid_range,
                k=wandb.config.spline_order,
                seed=wandb.config.seed,
                device=device).speed(compile=True, dynamic=False, fullgraph=True, options={"trace.enabled":True, "trace.graph_diagram":False, "triton.cudagraphs":True})

model(dataset['train_input'][[0]])

compiled_opt_kan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

checkpoint directory created: ./model
saving model version 0.0


/usr/local/lib/python3.11/dist-packages/torch/utils/_config_module.py:342: UserWarning: Skipping serialization of skipfiles_inline_module_allowlist value {}
  warnings.warn(
skipping cudagraphs due to skipping cudagraphs due to cpu device (primals_1). Found from : 
   File "/content/drive/MyDrive/optimized_kan/kan/MultKAN.py", line 785, in forward
    x = x[:,self.input_id.long()]

W0512 08:38:38.614000 171 torch/_inductor/debug.py:435] [0/2] model__2_forward_5 debug trace: /content/torch_compile_debug/run_2025_05_12_08_38_34_754515-pid_171/torchinductor/model__2_forward_5.0


setting lamb=0. If you want to set lamb > 0, set self.save_act=True


| train_loss: 1.44e-02 | test_loss: 7.22e-02 | reg: 0.00e+00 |: 100%|█| 1000/1000 [00:34<00:00, 29.0


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     120.551ms       161.46%     120.551ms      40.184ms           0 b           0 b           0 b           0 

/usr/local/lib/python3.11/dist-packages/torch/utils/_config_module.py:342: UserWarning: Skipping serialization of skipfiles_inline_module_allowlist value {}
  warnings.warn(
skipping cudagraphs due to skipping cudagraphs due to cpu device (arg0_1). Found from : 
   File "/content/drive/MyDrive/optimized_kan/kan/MultKAN.py", line 785, in forward
    x = x[:,self.input_id.long()]

W0512 08:39:25.995000 171 torch/_inductor/debug.py:435] [0/3] model__3_inference_7 debug trace: /content/torch_compile_debug/run_2025_05_12_08_38_34_754515-pid_171/torchinductor/model__3_inference_7.1


13.392978309


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▇▂▂▅▁▂▄▃▃▁▂▂▁▁▁▄▄▄▃▂▂▂▂▂▂▂▃▆█▆▆▆▇▅█▂▇▃▃▂
inference_time,▁
reg,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇███
train_loss,█▇▅▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),451.15491


In [35]:
run = wandb.init(
    entity=entity,
    project=project,
    name="Mixed Precision KAN",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = MPKAN(width=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid=wandb.config.grid_size,
                grid_range=wandb.config.grid_range,
                k=wandb.config.spline_order,
                seed=wandb.config.seed,
                device=device)

model(dataset['train_input'][[0]])

mpkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

/content/drive/MyDrive/optimized_kan/kan/MultKAN.py:819: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  self.subnode_actscale.append(torch.std(x, dim=0).detach())
/content/drive/MyDrive/optimized_kan/kan/MultKAN.py:829: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  input_range = torch.std(preacts, dim=0) + 0.1
/content/drive/MyDrive/optimized_kan/kan/MultKAN.py:830: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  output_range_spline = torch.std(postacts_numer

checkpoint directory created: ./model
saving model version 0.0


description:   0%|                                                         | 0/1000 [00:00<?, ?it/s]/content/drive/MyDrive/optimized_kan/kan/MixedKAN.py:52: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(opt == "Adam"))
/content/drive/MyDrive/optimized_kan/kan/MixedKAN.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
| train_loss: 4.34e-02 | test_loss: 7.59e-02 | reg: 7.06e+00 |: 100%|█| 1000/1000 [00:42<00:00, 23.3


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     171.775ms       232.97%     171.775ms      57.258ms           0 b           0 b           0 b           0 

<ipython-input-35-52f637edd68b>:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


54.35847280999997


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▅▁▂▂▁▂▁▂▁▄▁▅▄▅▅▂▁▁▂▁▂▁▁▁▁▂▄▃▆▅▂▁▂▁▂▂▁▂█▂
inference_time,▁
reg,██▇▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇████
train_loss,██▇▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▆▆▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),453.84472


In [42]:
# train the kan_model
# Initialize W&B project
run = wandb.init(
    entity=entity,
    project=project,
    name="Mixed Precision KAN Compiled",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = MPKAN(width=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid=wandb.config.grid_size,
                grid_range=wandb.config.grid_range,
                k=wandb.config.spline_order,
                seed=wandb.config.seed,
                device=device).speed(compile=True)

model(dataset['train_input'][[0]])

compiled_mpkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

checkpoint directory created: ./model
saving model version 0.0
setting lamb=0. If you want to set lamb > 0, set self.save_act=True


| train_loss: 1.47e-02 | test_loss: 7.19e-02 | reg: 0.00e+00 |: 100%|█| 1000/1000 [00:33<00:00, 30.2


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     114.003ms       167.82%     114.003ms      38.001ms           0 b           0 b           0 b           0 

<ipython-input-42-2f7be5a1a87a>:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
W0512 08:53:18.696000 171 torch/_dynamo/convert_frame.py:906] [0/8] torch._dynamo hit config.cache_size_limit (8)
W0512 08:53:18.696000 171 torch/_dynamo/convert_frame.py:906] [0/8]    function: 'forward' (/content/drive/MyDrive/optimized_kan/kan/MultKAN.py:751)
W0512 08:53:18.696000 171 torch/_dynamo/convert_frame.py:906] [0/8]    last reason: 0/0: GLOBAL_STATE changed: grad_mode 
W0512 08:53:18.696000 171 torch/_dynamo/convert_frame.py:906] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0512 08:53:18.696000 171 torch/_dynamo/convert_frame.py:906] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.
<ipython-input-42-2f7be5a1a87a>:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. 

6.935754696000004


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▁▃▂▁▃▂▂▃▂▁▃▄▄█▆▆▁▂▁▂▂▃▃▄▁▁▂▂▂▂▃▃▄▅▅▁▁▂▂▁
inference_time,▁
reg,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇████
train_loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),293.27787


In [43]:
run = wandb.init(
    entity=entity,
    project=project,
    name="Mixed Precision KAN Compiled w/ Options",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = MPKAN(width=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid=wandb.config.grid_size,
                grid_range=wandb.config.grid_range,
                k=wandb.config.spline_order,
                seed=wandb.config.seed,
                device=device).speed(compile=True, dynamic=False, fullgraph=True, options={"trace.enabled":True, "trace.graph_diagram":False, "triton.cudagraphs":True})

model(dataset['train_input'][[0]])

compiled_opt_mpkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

checkpoint directory created: ./model
saving model version 0.0
setting lamb=0. If you want to set lamb > 0, set self.save_act=True


description:   0%|                                                         | 0/1000 [00:00<?, ?it/s]/content/drive/MyDrive/optimized_kan/kan/MixedKAN.py:52: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(opt == "Adam"))
/content/drive/MyDrive/optimized_kan/kan/MixedKAN.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
| train_loss: 1.44e-02 | test_loss: 7.24e-02 | reg: 0.00e+00 |: 100%|█| 1000/1000 [00:38<00:00, 26.1


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     122.191ms       169.76%     122.191ms      40.730ms           0 b           0 b           0 b           0 

<ipython-input-43-4ed38a66eb1b>:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


7.9980498330000955


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▁▁▂▂▂▂▁▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▂▂▃▁▂▁▁▁▁▁▁▂▁▁▂▂█
inference_time,▁
reg,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
train_loss,█▇▇▇▅▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▇▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),314.29706


In [44]:
run = wandb.init(
    entity=entity,
    project=project,
    name="FastKAN",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = FastKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid_min=wandb.config.grid_range[0],
                grid_max=wandb.config.grid_range[1],
                num_grids=wandb.config.grid_size).to(device)


fastkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})


run.finish()

| train_loss: 1.53e-01 | test_loss: 6.47e-02 | : 100%|█████████| 1000/1000 [00:09<00:00, 107.34it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      30.680ms       253.80%      30.680ms      10.227ms           0 b           0 b           0 b           0 

cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,█▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇████
train_loss,█▇▇▇▆▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▆▅▅▄▄▄▄▄▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),116.93927
cuda_time_total (ms),12.08819


In [45]:
run = wandb.init(
    entity=entity,
    project=project,
    name="FastKAN Compiled",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = torch.compile(FastKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                              grid_min=wandb.config.grid_range[0],
                              grid_max=wandb.config.grid_range[1],
                              num_grids=wandb.config.grid_size
                              ).to(device),)

model(dataset['train_input'][[0]])

compiled_fastkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

| train_loss: 1.61e-01 | test_loss: 7.31e-02 | : 100%|██████████| 1000/1000 [00:10<00:00, 97.48it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      24.625ms       267.74%      24.625ms       8.208ms           0 b           0 b           0 b           0 

cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▂▁▁▃▁▁▁▂▁▂▂▁▃▂▃▁▁█▃▃▅▆▆▄▅▃▂▃▃▃▂▅▅▃▇▅▄▁▂▂
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇██
train_loss,█▆▆▅▅▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▅▅▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),93.61234
cuda_time_total (ms),9.19726


In [46]:
run = wandb.init(
    entity=entity,
    project=project,
    name="FastKAN Compiled w/ Options",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = torch.compile(FastKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                              grid_min=wandb.config.grid_range[0],
                              grid_max=wandb.config.grid_range[1],
                              num_grids=wandb.config.grid_size).to(device),
                      mode='max-autotune',
                      dynamic=False,
                      fullgraph=True,
                      )

model(dataset['train_input'][[0]])

compiled_opt_fastkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

| train_loss: 1.46e-01 | test_loss: 6.55e-02 | : 100%|█████████| 1000/1000 [00:09<00:00, 101.55it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      39.557ms       429.68%      39.557ms      13.186ms           0 b           0 b           0 b           0 

AUTOTUNE addmm(10000x11, 10000x5, 5x11)
  addmm 0.0127 ms 100.0% 
  bias_addmm 0.0199 ms 63.9% 
SingleProcess AUTOTUNE benchmarking takes 0.2256 seconds and 0.0002 seconds precompiling for 2 choices
AUTOTUNE addmm(10000x11, 10000x11, 11x11)
  addmm 0.0152 ms 100.0% 
  bias_addmm 0.0219 ms 69.6% 
SingleProcess AUTOTUNE benchmarking takes 0.2243 seconds and 0.0002 seconds precompiling for 2 choices


11.977098497999805


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▄▅█▅▆▆▂▂▂▁▂▄▅▂▂▂▂▂▁▂▂▂▁▂▁▅▂▄▂▄▁▂▂▂▁▁▁▂▁▃
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█
train_loss,█▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▇▆▅▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),150.64497
cuda_time_total (ms),9.20597


In [47]:
run = wandb.init(
    entity=entity,
    project=project,
    name="FastMPKAN",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = FastMPKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid_min=wandb.config.grid_range[0],
                grid_max=wandb.config.grid_range[1],
                num_grids=wandb.config.grid_size).to(device)

model(dataset['train_input'][[0]])

compiled_opt_fastmpkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})


run.finish()

/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
description:   0%|                                                         | 0/1000 [00:00<?, ?it/s]/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(opt == "Adam")):
| train_loss: 1.46e-01 | test_loss: 6.16e-02 | : 100%|██████████| 1000/1000 [00:12<00:00, 81.87it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      61.834ms       455.98%      61.834ms      20.611ms           0 b           0 b           0 b           0 

<ipython-input-47-6451a1d72ade>:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


1.2385788430001412


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,█▂▂▃▅▁▄▂▂▂▅▆▃▃▃▄▃▃▅▃▂▃▅▄█▃▁▁▁▁▂▃▂▂▂▂▁▁▂▁
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇██
train_loss,██▇▇▇▆▆▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▆▅▅▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),159.03544
cuda_time_total (ms),13.56062


In [48]:
run = wandb.init(
    entity=entity,
    project=project,
    name="FastMPKAN Compiled",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = torch.compile(FastMPKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                grid_min=wandb.config.grid_range[0],
                grid_max=wandb.config.grid_range[1],
                num_grids=wandb.config.grid_size).to(device))
model(dataset['train_input'][[0]])
fastmpkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})


run.finish()

/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
description:   0%|                                                         | 0/1000 [00:00<?, ?it/s]/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(opt == "Adam")):
| train_loss: 1.47e-01 | test_loss: 6.42e-02 | : 100%|██████████| 1000/1000 [00:12<00:00, 82.47it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      49.273ms       363.12%      49.273ms      16.424ms           0 b           0 b           0 b           0 

<ipython-input-48-2754951e6450>:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
W0512 08:58:24.439000 171 torch/_dynamo/convert_frame.py:906] [1/8] torch._dynamo hit config.cache_size_limit (8)
W0512 08:58:24.439000 171 torch/_dynamo/convert_frame.py:906] [1/8]    function: 'forward' (/content/drive/MyDrive/optimized_kan/kan/FastKAN.py:133)
W0512 08:58:24.439000 171 torch/_dynamo/convert_frame.py:906] [1/8]    last reason: 1/0: GLOBAL_STATE changed: grad_mode 
W0512 08:58:24.439000 171 torch/_dynamo/convert_frame.py:906] [1/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0512 08:58:24.439000 171 torch/_dynamo/convert_frame.py:906] [1/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.
<ipython-input-48-2754951e6450>:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. 

1.3156452309999622


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▅▂▂▂▂▁▂▂▃▂▂▄█▅▄▂▂▁▂▂▁▁▁▁▁▁▂▁▁▁▁▁▂▁▁▁▁▂▂▁
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train_loss,█▇▆▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),130.81666
cuda_time_total (ms),13.56951


In [49]:
run = wandb.init(
    entity=entity,
    project=project,
    name="FastMPKAN Compiled w/ Options",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = torch.compile(FastMPKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                              grid_min=wandb.config.grid_range[0],
                              grid_max=wandb.config.grid_range[1],
                              num_grids=wandb.config.grid_size).to(device),
                      dynamic=False,
                      fullgraph=True,
                      options={"trace.enabled":True, "trace.graph_diagram":False, "triton.cudagraphs":True})
model(dataset['train_input'][[0]])
compiled_opt_fastmpkan_metrics = model.fit(dataset,
                            opt=wandb.config.optimizer,
                            batch=wandb.config.batch_size,
                            steps=wandb.config.training_steps,
                            lr=wandb.config.lr,
                            lamb=0.001,
                            profile=True)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
description:   0%|                                                         | 0/1000 [00:00<?, ?it/s]/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/content/drive/MyDrive/optimized_kan/kan/FastMixedKAN.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(opt == "Adam")):
| train_loss: 1.49e-01 | test_loss: 6.34e-02 | : 100%|██████████| 1000/1000 [00:11<00:00, 86.52it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      70.396ms       519.24%      70.396ms      23.465ms           0 b           0 b           0 b           0 

<ipython-input-49-65c304ff2bb6>:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


1.7570672859999377


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▃█▄▇▆▂▂▂▂▁▂▂▁▂▂▁▂▄▄▂▁▂▃▂▃▁▂▂▁▂▁▂▁▇▂▂▂▁▄▂
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇█
train_loss,██▆▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▅▅▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),188.24939
cuda_time_total (ms),13.55756


In [51]:
# Initialize quantized model
try:
    run.finish()
except:
    pass
run = wandb.init(
    entity=entity,
    project=project,
    name="QuantFastKAN",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = QuantFastKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                              grid_min=wandb.config.grid_range[0],
                              grid_max=wandb.config.grid_range[1],
                              num_grids=wandb.config.grid_size).to(device)

for module in model.modules():
    if isinstance(module, (QuantSplineLinear, QuantRadialBasisFunction)):
        module.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')
model = torch.ao.quantization.prepare_qat(model)

model(dataset['train_input'])

model.fit(dataset,
        opt=wandb.config.optimizer,
        batch=wandb.config.batch_size,
        steps=wandb.config.training_steps,
        lr=wandb.config.lr,
        lamb=0.001,
       profile=True)

model.eval()
# model = torch.ao.quantization.convert(model)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▂▂▁▂▂▂█▄▃▂▂▃▂▂▂▃▃▄▂▂▂▁▂▇▂▃▂▃▄▃▂▂▂▂▁▁▂▄▄▄
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▁▂▂▂▂▂▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
train_loss,█▇▅▅▅▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▇▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),120.76767
cuda_time_total (ms),16.32436
exec_time,0.01206


| train_loss: 1.51e-01 | test_loss: 6.32e-02 | : 100%|██████████| 1000/1000 [00:15<00:00, 66.01it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      35.481ms       217.22%      35.481ms      11.827ms           0 b           0 b           0 b           0 

cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▁▁▁▁▂▂▁▁▂▂▃▁▄▃▃▁▁▅▆█▃▃▆▂▁▂▂▄▂▃▃▂▂▂▃▁▁▁▁▁
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
train_loss,█▇▆▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▆▅▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),131.5291
cuda_time_total (ms),16.33377


In [52]:
# Initialize quantized model
try:
    run.finish()
except:
    pass
run = wandb.init(
    entity=entity,
    project=project,
    name="QuantFastKAN Compiled",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = QuantFastKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                              grid_min=wandb.config.grid_range[0],
                              grid_max=wandb.config.grid_range[1],
                              num_grids=wandb.config.grid_size).to(device)

for module in model.modules():
    if isinstance(module, (QuantSplineLinear, QuantRadialBasisFunction)):
        module.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')

# model = torch.ao.quantization.prepare_qat(model)

model = torch.compile(model)

model.fit(dataset,
        opt=wandb.config.optimizer,
        batch=wandb.config.batch_size,
        steps=wandb.config.training_steps,
        lr=wandb.config.lr,
        lamb=0.001,
       profile=True)

model.eval()
model = torch.ao.quantization.convert(model)

model(dataset['train_input'])
import time
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

| train_loss: 1.48e-01 | test_loss: 5.90e-02 | : 100%|██████████| 1000/1000 [00:10<00:00, 92.44it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      29.102ms       316.53%      29.102ms       9.701ms           0 b           0 b           0 b           0 

cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,▂▁▂▂▂▂▂▂▂▂▂▂▁▂▂▁▄▅▄▂▂▄▂█▇▄▅▅▆▃▄▁▁▂▁▂▁▂▃▂
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train_loss,█▇▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▆▆▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),109.97138
cuda_time_total (ms),9.19423


In [64]:
# Initialize quantized model
try:
    run.finish()
except:
    pass
run = wandb.init(
    entity=entity,
    project=project,
    name="QuantFastKAN Compiled w/ Options",
    config={
    "equation": "Black-Scholes Option Pricing",
    "n_var": n_var,
    "train_num": train_num,
    "test_num": test_num,

    "input_dim": n_var,
    "hidden_dim": hidden_dim,
    "out_dim": out_dim,
    "grid_size": grid_size,
    "grid_range": grid_range,
    "spline_order":spline_order,
    "seed": seed,

    "optimizer": opt,
    "batch_size": batch_size,
    "training_steps": training_steps,
    "lr": lr,
})

model = QuantFastKAN(layers_hidden=[wandb.config.input_dim, *wandb.config.hidden_dim, wandb.config.out_dim],
                              grid_min=wandb.config.grid_range[0],
                              grid_max=wandb.config.grid_range[1],
                              num_grids=wandb.config.grid_size).to(device)

for module in model.modules():
    if isinstance(module, (QuantSplineLinear, QuantRadialBasisFunction)):
        module.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')

# model = torch.ao.quantization.prepare_qat(model)

model = torch.compile(model,mode='max-autotune'
                      )

model.fit(dataset,
        opt=wandb.config.optimizer,
        batch=wandb.config.batch_size,
        steps=wandb.config.training_steps,
        lr=wandb.config.lr,
        lamb=0.001,
       profile=True)



# model = torch.ao.quantization.convert(model)

model(dataset['train_input'])
import time
import torch._dynamo
torch._dynamo.config.suppress_errors = True
start_time = time.perf_counter()

for _ in range(1000):
    model.eval()
    with torch.no_grad():
        model(dataset['test_input'])

end_time = time.perf_counter()
print(end_time - start_time)
wandb.log({"inference_time": end_time - start_time})

run.finish()

| train_loss: 1.46e-01 | test_loss: 6.89e-02 | : 100%|██████████| 1000/1000 [00:10<00:00, 97.72it/s]


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      27.883ms       230.25%      27.883ms       9.294ms           0 b           0 b           0 b           0 

W0512 09:12:29.535000 171 torch/_dynamo/convert_frame.py:906] [3/17] torch._dynamo hit config.cache_size_limit (8)
W0512 09:12:29.535000 171 torch/_dynamo/convert_frame.py:906] [3/17]    function: 'forward' (/content/drive/MyDrive/optimized_kan/kan/QuantFastKAN.py:85)
W0512 09:12:29.535000 171 torch/_dynamo/convert_frame.py:906] [3/17]    last reason: 3/0: GLOBAL_STATE changed: grad_mode 
W0512 09:12:29.535000 171 torch/_dynamo/convert_frame.py:906] [3/17] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0512 09:12:29.535000 171 torch/_dynamo/convert_frame.py:906] [3/17] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


1.1205175860000054


cpu_time_total (ms),▁
cuda_time_total (ms),▁
exec_time,█▂▃▅▂▂▁▁▁▁▂▂▂▁▁▄▁▂▂▃▂▁▂▁▁▁▁▁▃▁▂▂▁▁▁▂▁▁▂▂
inference_time,▁
self_cpu_time_total (ms),▁
self_cuda_time_total (ms),▁
step,▁▁▁▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▇▇▇▇▇▇▇█
train_loss,█▇▇▇▆▆▆▆▄▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▆▄▄▄▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
cpu_time_total (ms),101.75578
cuda_time_total (ms),12.10994


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(np.array(kan_metrics['train_loss']), label='KAN Train Loss')
plt.plot(np.array(kan_metrics['test_loss']), label='KAN Test Loss')
plt.plot(np.array(mpkan_metrics['train_loss']), label='MPKAN Train Loss')
plt.plot(np.array(mpkan_metrics['test_loss']), label='MPKAN Test Loss')
plt.plot(np.array(compiled_kan_metrics['train_loss']), label='Compiled KAN Train Loss')
plt.plot(np.array(compiled_kan_metrics['test_loss']), label='Compiled KAN Test Loss')
plt.plot(np.array(compiled_mpkan_metrics['train_loss']), label='Compiled MPKAN Train Loss')
plt.plot(np.array(compiled_mpkan_metrics['test_loss']), label='Compiled MPKAN Test Loss')

plt.plot(np.array(fastkan_metrics['train_loss']), label='FastKAN Train Loss')
plt.plot(np.array(fastkan_metrics['test_loss']), label='FastKAN Test Loss')

plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training and Test Loss over Iteration')
plt.legend()  # No arguments needed; uses the labels above
plt.show()
